# Fase 3 - Validacion de Dataset PyTorch

Notebook base para comprobar que los CSV generados en Fase 2 alimentan correctamente `NIHMultilabelDataset` y un `DataLoader`. No entrena modelos.

## 1. Cargar repo

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

# Si el repo no esta en el entorno, descomenta y ajusta la URL:
# !git clone https://github.com/USUARIO/Tesis_CXR.git /kaggle/working/Tesis_CXR

candidate_repos = [Path.cwd(), Path('/kaggle/working/Tesis_CXR')]
REPO_DIR = next((path for path in candidate_repos if (path / 'scripts').exists()), None)

if REPO_DIR is None:
    raise FileNotFoundError('No se encontro el repositorio. Clona Tesis_CXR en /kaggle/working/Tesis_CXR.')

print('Repo:', REPO_DIR)

## 2. Cargar labels.json

In [ ]:
LABELS_JSON = REPO_DIR / 'artifacts' / 'labels.json'
with LABELS_JSON.open('r', encoding='utf-8') as file:
    LABELS = json.load(file)

assert len(LABELS) == 14
assert 'No Finding' not in LABELS
LABELS

## 3. Usar train.csv procesado

In [ ]:
TRAIN_CSV = Path('/kaggle/working/processed/splits/train.csv')
if not TRAIN_CSV.exists():
    raise FileNotFoundError('No existe /kaggle/working/processed/splits/train.csv. Ejecuta primero Fase 2.')

train_df = pd.read_csv(TRAIN_CSV)
print('Rows:', len(train_df))
print('Patients:', train_df['Patient ID'].astype(str).nunique())
assert 'No Finding' not in train_df.columns

## 4. Ejecutar scripts/check_dataset.py

In [ ]:
check_cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts' / 'check_dataset.py'),
    '--csv-path', str(TRAIN_CSV),
    '--labels-json', str(LABELS_JSON),
    '--batch-size', '8',
    '--num-workers', '2',
]

subprocess.run(check_cmd, check=True)

## 5. Visualizar una imagen con etiquetas positivas

In [ ]:
sample = train_df.iloc[0]
positive_labels = [label for label in LABELS if int(sample[label]) == 1]
title = ', '.join(positive_labels) if positive_labels else 'No positive pathology labels'

image = Image.open(sample['image_path']).convert('RGB')
plt.figure(figsize=(6, 6))
plt.imshow(image, cmap='gray')
plt.axis('off')
plt.title(title)
plt.show()

{
    'image_index': sample.get('Image Index'),
    'patient_id': sample.get('Patient ID'),
    'image_path': sample.get('image_path'),
    'positive_labels': positive_labels,
}

## 6. Mostrar distribucion simple de etiquetas

In [ ]:
label_counts = train_df[LABELS].sum().astype(int).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
label_counts.plot(kind='barh')
plt.xlabel('Positive rows')
plt.title('NIH train split label distribution')
plt.tight_layout()
plt.show()

label_counts.sort_values(ascending=False)